In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets_dataset_2.csv
/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets.csv


In [2]:
col_to_use = ['date','user_name','text','user_followers', 'is_retweet']

df0 = pd.read_csv('/kaggle/input/datasets/kaushiksuresh147/bitcoin-tweets/Bitcoin_tweets.csv',
                  chunksize=100000,
                  usecols=col_to_use,
                  engine='python',
                  on_bad_lines='skip')
df = pd.concat(df0, ignore_index=True)

In [3]:
df.head(10)

,user_name,user_followers,date,text,is_retweet
0,DeSota Wilson,8534.0,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,False
1,CryptoND,6769.0,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""...",False
2,Tdlmatias,128.0,2021-02-10 23:54:48,"Guys evening, I have read this article about B...",False
3,Crypto is the future,625.0,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,False
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,False
5,ZerrBenz™ ⚔ ✪ 20732,742.0,2021-02-10 23:53:30,💹 Trade #Crypto on #Binance \n\n📌 Enjoy #Cashb...,False
6,Bitcoin-Bot,131.0,2021-02-10 23:53:17,&lt;'fire' &amp; 'man'&gt;\n#Bitcoin #Crypto #...,False
7,Cryptocurrencies / EUR,4052.0,2021-02-10 23:52:42,🔄 Prices update in $EUR (1 hour):\n\n$BTC - ...,False
8,Mikcoin,104.0,2021-02-10 23:52:25,#BTC #Bitcoin #Ethereum #ETH #Crypto #cryptotr...,False
9,DeSota Wilson,8534.0,2021-02-10 23:52:08,.@Tesla’s #bitcoin investment is revolutionary...,False


In [4]:
df.tail(10)

,user_name,user_followers,date,text,is_retweet
4693081,世界杯下注,1.0,2023-01-06 17:46:53,Jonas Cumberland #btc #彩票 Merle Noyes #世界杯直播 h...,False
4693082,Gale Browning,33.0,2023-01-06 17:46:50,❤️ Join me at Bybit and earn exclusive rewards...,False
4693083,Jeanie Phillips,22.0,2023-01-06 17:46:50,❤️ Join me at Bybit and earn exclusive rewards...,False
4693084,世界杯投注平台,4.0,2023-01-06 17:46:44,Lorraine Harvey #btc #彩票 Coral Bart #世界杯直播 htt...,False
4693085,Behzad.moni,66.0,2023-01-06 17:46:36,#DogelonMars is the future. #TSUKA is the nex...,False
4693086,TAnotepad,674.0,2023-01-06 17:46:35,"Bitcoin squeeze is SUPER TIGHT, which way will...",False
4693087,Boba-Feh,79.0,2023-01-06 17:46:29,Closed #BTC short at 16725. Missed my long pla...,False
4693088,Ethereum Yoda,532.0,2023-01-06 17:46:22,#Ethereum price update: \n\n#ETH $1263.59 USD\...,False
4693089,Bitcoin Price Ticker,83.0,2023-01-06 17:46:20,1₿ = $16814.7 -0.07%🔻\n\nDetails:\nChange: 🔻-1...,False
4693090,faucetojisan,14.0,2023-01-06 17:46:17,Earn crypto by playing fun games online.\nGet ...,False


In [5]:
df.shape

(4693091, 5)

# Text preprocessing

Lọc nhiễu, xóa spam, bot

In [6]:
import re
import pandas as pd

# =====================================================================
# COMPILE REGEX TOÀN CỤC (GLOBAL) - CHỈ BỊ BIÊN DỊCH 1 LẦN DUY NHẤT
# =====================================================================
# Tinh chỉnh danh sách từ khóa bot để tránh quét nhầm các tài khoản tin tức/KOLs uy tín
BOT_NAME_KEYWORDS = ['bot', 'ticker', 'tracker', 'faucet', 'robot', 'automated']
BOT_NAME_REGEX = re.compile('|'.join(BOT_NAME_KEYWORDS), flags=re.IGNORECASE)

AUTO_TWEET_PATTERNS = [
    r'^\d+\s*\u20bf?\s*=\s*\$[\d,\.]+',     # Ví dụ: '1\u20bf = $16814.7...'
    r'^.{0,15}prices?\s+update',             # Ví dụ: 'Prices update in $EUR...'
    r'\u2764\ufe0f\s*join me at',            # Link giới thiệu (referral spam)
    r'earn crypto by playing',               # Game kiếm tiền / faucet spam
    r'join me at bybit',                     # Bybit referral
]
AUTO_TWEET_REGEX = re.compile('|'.join(AUTO_TWEET_PATTERNS), flags=re.IGNORECASE)


def filter_tweets(df, min_text_length=10, spam_tweet_threshold=50,
                  spam_window_hours=1, report=True):
    """
    Bước 1 — Lọc dữ liệu thô, loại bỏ Retweets, Spam và Bot.
    """
    df = df.copy()
    initial_count = len(df)
    stats = {}

    # 0. Ép kiểu dữ liệu chuỗi và thời gian ngay từ đầu để tránh lỗi NaN/Mismatch
    df['text'] = df['text'].astype(str).str.strip()
    df['user_name'] = df['user_name'].astype(str).str.strip()
    
    # Đồng bộ hóa múi giờ sang UTC để tránh lỗi lệch múi giờ khi gom cụm
    df['date'] = pd.to_datetime(df['date'], errors='coerce', utc=True)
    df = df.dropna(subset=['date'])

    # 1. Loại hàng trùng lặp (trên text thô và user_name)
    before = len(df)
    df = df.drop_duplicates(subset=['text', 'user_name'])
    stats['duplicates'] = before - len(df)

    # 2. Loại retweet (kết hợp cả metadata cột và quét tiền tố text để tránh bỏ sót)
    before = len(df)
    is_rt_metadata = False
    if 'is_retweet' in df.columns:
        is_rt_metadata = df['is_retweet'].astype(str).str.lower() == 'true'
        
    is_rt_prefix = df['text'].str.startswith('RT @', na=False)
    
    # Loại bỏ nếu vi phạm một trong hai điều kiện
    df = df[~(is_rt_metadata | is_rt_prefix)]
    stats['retweets'] = before - len(df)

    # 3. Loại tweet quá ngắn / quá dài
    before = len(df)
    text_len = df['text'].str.len()
    df = df[(text_len >= min_text_length) & (text_len <= 1000)]
    stats['invalid_text'] = before - len(df)

    # 4. Loại bot account dựa vào user_name (Sử dụng Regex precompiled)
    before = len(df)
    is_bot = df['user_name'].str.contains(BOT_NAME_REGEX, na=False)
    df = df[~is_bot]
    stats['bots'] = before - len(df)

    # 5. Loại tweet tự động / spam dựa vào nội dung text (Sử dụng Regex precompiled)
    before = len(df)
    is_auto = df['text'].str.contains(AUTO_TWEET_REGEX, na=False)
    df = df[~is_auto]
    stats['auto_tweets'] = before - len(df)

    # 6. Loại spam account (Đăng quá nhiều tweet trong cùng 1 cửa sổ giờ)
    before = len(df)
    # dt.floor hoạt động rất chuẩn xác khi date đã được đồng bộ UTC ở bước 0
    df['_time_window'] = df['date'].dt.floor(f'{spam_window_hours}h')
    
    tweet_counts = (
        df.groupby(['user_name', '_time_window'])
          .size()
          .reset_index(name='_count')
    )
    
    # Xác định các user vượt ngưỡng spam
    spam_users = tweet_counts[
        tweet_counts['_count'] > spam_tweet_threshold
    ]['user_name'].unique()
    
    # Loại bỏ hoàn toàn các user này khỏi tập dữ liệu
    df = df[~df['user_name'].isin(spam_users)].drop(columns=['_time_window'])
    stats['spam_accounts'] = before - len(df)

    # Báo cáo tổng kết
    final_count = len(df)
    total_removed = initial_count - final_count
    retention = (final_count / initial_count) * 100 if initial_count > 0 else 0

    if report:
        print('=' * 50)
        print('    BÁO CÁO LỌC DỮ LIỆU — BƯỚC 1 (ĐÃ TỐI ƯU)')
        print('=' * 50)
        print(f"  Tweet ban đầu         : {initial_count:>10,}")
        print(f"  Trùng lặp loại bỏ     : {stats['duplicates']:>10,}")
        print(f"  Retweet loại bỏ       : {stats['retweets']:>10,}")
        print(f"  Text bất thường       : {stats['invalid_text']:>10,}")
        print(f"  Bot account           : {stats['bots']:>10,}")
        print(f"  Tweet tự động / spam  : {stats['auto_tweets']:>10,}")
        print(f"  Spam account          : {stats['spam_accounts']:>10,}")
        print('-' * 50)
        print(f"  Tổng loại bỏ          : {total_removed:>10,}")
        print(f"  Tweet còn lại         : {final_count:>10,}")
        print(f"  Tỉ lệ giữ lại         : {retention:>9.1f}%")
        print('=' * 50)

    return df.reset_index(drop=True)


# Thực thi Bước 1
df = filter_tweets(df)

    BÁO CÁO LỌC DỮ LIỆU — BƯỚC 1 (ĐÃ TỐI ƯU)
  Tweet ban đầu         :  4,693,091
  Trùng lặp loại bỏ     :        479
  Retweet loại bỏ       :          0
  Text bất thường       :        115
  Bot account           :    146,567
  Tweet tự động / spam  :     36,823
  Spam account          :    479,027
--------------------------------------------------
  Tổng loại bỏ          :    666,814
  Tweet còn lại         :  4,026,277
  Tỉ lệ giữ lại         :      85.8%


Chuyển emoji thành text

In [7]:
!pip install emoji

In [8]:
# Cài đặt thư viện emoji nếu chưa có (bỏ dấu comment '#' phía dưới để chạy nếu cần)
# !pip install emoji --upgrade

import re
import pandas as pd

# --- KIỂM TRA VÀ IMPORT THƯ VIỆN EMOJI ---
try:
    import emoji
    HAS_EMOJI_LIB = True
except ImportError:
    HAS_EMOJI_LIB = False
    # Từ điển fallback chứa các emoji cực kỳ phổ biến trong thị trường crypto
    # (Đảm bảo code chạy mượt mà ngay cả khi môi trường chưa cài thư viện emoji)
    CRYPTO_EMOJI_DICT = {
        '🚀': ' rocket ', '💎': ' gemstone ', '🙌': ' raising hands ', 
        '🔥': ' fire ', '📈': ' chart increasing ', '📉': ' chart decreasing ',
        '🐻': ' bear ', '🐂': ' bull ', '🌕': ' full moon ', '🌙': ' crescent moon ',
        '💀': ' skull ', '💩': ' pile of poo ', '🤡': ' clown face ', 
        '💯': ' hundred points ', '🙏': ' folded hands ', '👍': ' thumbs up ',
        '👎': ' thumbs down ', '😍': ' smiling face with heart eyes ',
        '🤔': ' thinking face ', '😭': ' loudly crying face ',
        '🤣': ' rolling on the floor laughing ', '😂': ' face with tears of joy '
    }


def convert_emojis(text: str) -> str:
    """
    Bước 2 — Chuyển đổi Emoji thành văn bản tiếng Anh tương ứng.
    (Chạy ở cell riêng trước khi xóa các ký tự đặc biệt để không bị mất Emoji)
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    
    if HAS_EMOJI_LIB:
        # Demojize chuyển emoji sang dạng :emoji_name: (ví dụ: 🚀 -> :rocket:)
        text_demo = emoji.demojize(text)
        # Thay thế dấu gạch dưới thành khoảng trắng và loại bỏ dấu hai chấm
        # Ví dụ: :smiling_face_with_smiling_eyes: -> smiling face with smiling eyes
        def replace_emoji_name(match):
            emoji_name = match.group(1)
            return " " + emoji_name.replace('_', ' ') + " "
        return re.sub(r':([a-zA-Z0-9_-]+):', replace_emoji_name, text_demo)
    else:
        # Dùng dictionary fallback nếu môi trường chưa cài thư viện emoji
        for emo, text_rep in CRYPTO_EMOJI_DICT.items():
            text = text.replace(emo, text_rep)
        return text


def apply_emoji_conversion(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng chuyển đổi emoji lên DataFrame, khởi tạo cột làm sạch 'text_clean'.
    """
    df = df.copy()
    
    # Thực hiện dịch Emoji trên cột text gốc, lưu vào cột trung gian text_clean
    df['text_clean'] = df['text'].apply(convert_emojis)
    
    if report:
        print('=' * 65)
        print('    BÁO CÁO DỊCH EMOJI SANG CHỮ TIẾNG ANH — BƯỚC 2')
        print('=' * 65)
        print(f'  Số lượng tweet đã xử lý : {len(df):>10,}')
        print('=' * 65)
        print('\n  Ví dụ mẫu tweet sau khi dịch Emoji:')
        samples = df[['text', 'text_clean']].head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Gốc : {row["text"][:120]}')
            print(f'      Sau : {row["text_clean"][:120]}')
        print('=' * 65)
        
    return df


# Thực thi Bước 2
df = apply_emoji_conversion(df)

    BÁO CÁO DỊCH EMOJI SANG CHỮ TIẾNG ANH — BƯỚC 2
  Số lượng tweet đã xử lý :  4,026,277

  Ví dụ mẫu tweet sau khi dịch Emoji:

  [0] Gốc : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.co/xaaZmaJKiV @MyBlueRidgeBank… https://
      Sau : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.co/xaaZmaJKiV @MyBlueRidgeBank… https://

  [1] Gốc : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWandersleb, #Btc #wallet #security expe… htt
      Sau :  smiling face with sunglasses  Today, that's this #Thursday, we will do a " clapper board  Take 2" with our friend @LeoW

  [2] Gốc : Guys evening, I have read this article about BTC and would like to share with you all - https://t.co/QxCZgmuy3B… https:/
      Sau : Guys evening, I have read this article about BTC and would like to share with you all - https://t.co/QxCZgmuy3B… https:/


In [9]:
df.shape

(4026277, 6)

3 - Làm sạch URL, Mentions, Nháy cong & Xử lý Hashtag

In [10]:
import html
import re
import pandas as pd

# =====================================================================
# THIẾT LẬP REGEX TOÀN CỤC & HASHTAG THỊ TRƯỜNG CỐT LÕI
# =====================================================================
URL_PATTERN = re.compile(r'https?://\S+|www\.\S+', flags=re.IGNORECASE)
MENTION_PATTERN = re.compile(r'@\w+')
HASHTAG_PATTERN = re.compile(r'#(\w+)')

# Regex phục vụ việc tách từ dạng CamelCase cho hashtag
CAMEL_CASE_1 = re.compile(r'([a-z])([A-Z])')
CAMEL_CASE_2 = re.compile(r'([A-Z]+)([A-Z][a-z])')

# Tập hợp các hashtag liên quan trực tiếp đến thị trường cần GIỮ NGUYÊN cấu trúc '#'
MARKET_HASHTAGS = {
    'btc', 'bitcoin', 'eth', 'ethereum', 'crypto', 'cryptocurrency', 
    'bnb', 'sol', 'ada', 'xrp', 'doge', 'altcoin', 'bullrun'
}


def expand_hashtag(match) -> str:
    """
    Xử lý Hashtag theo yêu cầu:
    - Giữ nguyên # đối với các hashtag thị trường cốt lõi (ví dụ: #BTC, #Bitcoin)
    - Tách CamelCase và xóa # đối với các hashtag khác (ví dụ: #BitcoinPump -> Bitcoin Pump)
    """
    tag = match.group(1)
    tag_lower = tag.lower()
    
    if tag_lower in MARKET_HASHTAGS:
        return f"#{tag}"
    else:
        # Tách PascalCase / camelCase
        expanded = CAMEL_CASE_1.sub(r'\1 \2', tag)
        expanded = CAMEL_CASE_2.sub(r'\1 \2', expanded)
        return " " + expanded + " "


def clean_text_structure(text: str) -> str:
    """
    Bước 3 — Làm sạch URL, Mentions, chuẩn hóa nháy cong & Xử lý phân loại Hashtag.
    """
    if not isinstance(text, str) or not text.strip():
        return ''
    
    # 1. Giải mã HTML entities (ví dụ: &amp; -> &)
    text = html.unescape(text)
    
    # 2. Xóa URL
    text = URL_PATTERN.sub('', text)
    
    # 3. Xóa @mention
    text = MENTION_PATTERN.sub('', text)
    
    # 4. Xử lý Hashtag
    text = HASHTAG_PATTERN.sub(expand_hashtag, text)
    
    # 5. Chuẩn hóa dấu nháy cong thành nháy thẳng
    text = text.replace('’', "'").replace('‘', "'")
    
    return text


def apply_text_cleaning(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Áp dụng làm sạch cấu trúc lên cột 'text_clean' đã có emoji của Bước 2.
    """
    df = df.copy()
    
    # Áp dụng hàm làm sạch cấu trúc lên cột text_clean
    df['text_clean'] = df['text_clean'].apply(clean_text_structure)
    
    # Chuẩn hóa các khoảng trắng thừa sinh ra trong quá trình làm sạch
    df['text_clean'] = df['text_clean'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    if report:
        print('=' * 65)
        print('    BÁO CÁO LÀM SẠCH CẤU TRÚC & HASHTAG — BƯỚC 3')
        print('=' * 65)
        print(f'  Số lượng tweet đã xử lý : {len(df):>10,}')
        print('=' * 65)
        print('\n  Ví dụ mẫu tweet sau khi làm sạch cấu trúc:')
        samples = df[['text', 'text_clean']].head(3)
        for i, row in samples.iterrows():
            print(f'\n  [{i}] Gốc : {row["text"][:120]}')
            print(f'      Sạch: {row["text_clean"][:120]}')
        print('=' * 65)
        
    return df


# Thực thi Bước 3
df = apply_text_cleaning(df)

    BÁO CÁO LÀM SẠCH CẤU TRÚC & HASHTAG — BƯỚC 3
  Số lượng tweet đã xử lý :  4,026,277

  Ví dụ mẫu tweet sau khi làm sạch cấu trúc:

  [0] Gốc : Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement https://t.co/xaaZmaJKiV @MyBlueRidgeBank… https://
      Sạch: Blue Ridge Bank shares halted by NYSE after #bitcoin ATM announcement …

  [1] Gốc : 😎 Today, that's this #Thursday, we will do a "🎬 Take 2" with our friend @LeoWandersleb, #Btc #wallet #security expe… htt
      Sạch: smiling face with sunglasses Today, that's this Thursday , we will do a " clapper board Take 2" with our friend , #Btc w

  [2] Gốc : Guys evening, I have read this article about BTC and would like to share with you all - https://t.co/QxCZgmuy3B… https:/
      Sạch: Guys evening, I have read this article about BTC and would like to share with you all -


In [11]:
df.shape

(4026277, 6)

In [12]:
df.head()

,user_name,user_followers,date,text,is_retweet,text_clean
0,DeSota Wilson,8534.0,2021-02-10 23:59:04+00:00,Blue Ridge Bank shares halted by NYSE after #b...,False,Blue Ridge Bank shares halted by NYSE after #b...
1,CryptoND,6769.0,2021-02-10 23:58:48+00:00,"😎 Today, that's this #Thursday, we will do a ""...",False,"smiling face with sunglasses Today, that's thi..."
2,Tdlmatias,128.0,2021-02-10 23:54:48+00:00,"Guys evening, I have read this article about B...",False,"Guys evening, I have read this article about B..."
3,Crypto is the future,625.0,2021-02-10 23:54:33+00:00,$BTC A big chance in a billion! Price: \487264...,False,$BTC A big chance in a billion! Price: \487264...
4,Alex Kirchmaier 🇦🇹🇸🇪 #FactsSuperspreader,1249.0,2021-02-10 23:54:06+00:00,This network is secured by 9 508 nodes as of t...,False,This network is secured by 9 508 nodes as of t...


4 - Lọc trùng lặp sâu

In [13]:
import pandas as pd

def deep_deduplicate(df: pd.DataFrame, report: bool = True) -> pd.DataFrame:
    """
    Bước 4 — Lọc trùng lặp sâu (Deep Semantic Deduplication).
    
    Thao tác lọc trùng lặp dựa trên cột 'text_clean' đã làm sạch hoàn toàn ở Bước 2 & 3.
    Loại bỏ triệt để các tweet có nội dung ngữ nghĩa giống hệt nhau nhưng ban đầu 
    chỉ khác nhau ở đường link URL hoặc tag mention (thường do bot spam).
    """
    df = df.copy()
    before_count = len(df)
    
    # Thực hiện lọc trùng lặp sâu dựa trên văn bản cốt lõi 'text_clean'
    # Giữ lại bản ghi xuất hiện đầu tiên (keep='first')
    df = df.drop_duplicates(subset=['text_clean'], keep='first')
    
    dropped_count = before_count - len(df)
    retention_rate = (len(df) / before_count) * 100 if before_count > 0 else 0
    
    if report:
        print('=' * 65)
        print('    BÁO CÁO LỌC TRÙNG LẶP SÂU (SEMANTIC DEDUPLICATION) — BƯỚC 4')
        print('=' * 65)
        print(f'  Tweet đầu vào trước khi lọc sâu    : {before_count:>10,}')
        print(f'  Tweet trùng lặp ngữ nghĩa bị loại  : {dropped_count:>10,}')
        print(f'  Tweet thực tế còn lại              : {len(df):>10,}')
        print(f'  Tỉ lệ giữ lại của bước này         : {retention_rate:>9.1f}%')
        print('=' * 65)
        
    return df.reset_index(drop=True)


# Thực thi Bước 4
df = deep_deduplicate(df)

    BÁO CÁO LỌC TRÙNG LẶP SÂU (SEMANTIC DEDUPLICATION) — BƯỚC 4
  Tweet đầu vào trước khi lọc sâu    :  4,026,277
  Tweet trùng lặp ngữ nghĩa bị loại  :    726,623
  Tweet thực tế còn lại              :  3,299,654
  Tỉ lệ giữ lại của bước này         :      82.0%


In [14]:
df.to_csv("btc_tweet_buoc4.csv",index = False)

hết bước 4

In [15]:
# Chia df thành danh sách chứa 10 DataFrame con
df_list = np.array_split(df, 10)

# Truy cập từng df: df_list[0], df_list[1], ..., df_list[9]
# Ví dụ: gán ra các biến riêng lẻ nếu muốn
df1, df2, df3, df4, df5, df6, df7, df8, df9, df10 = df_list

print(f"Số dòng của df1: {len(df1):,}")

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Số dòng của df1: 329,966


In [16]:
print(df1.shape)
print(df2.shape)
print(df3.shape)
print(df4.shape)
print(df5.shape)
print(df6.shape)
print(df7.shape)
print(df8.shape)
print(df9.shape)
print(df10.shape)

(329966, 6)
(329966, 6)
(329966, 6)
(329966, 6)
(329965, 6)
(329965, 6)
(329965, 6)
(329965, 6)
(329965, 6)
(329965, 6)


In [17]:
df1.to_csv('btc_tweet_1.csv',index=False, encoding='utf-8-sig')
df2.to_csv('btc_tweet_2.csv',index=False, encoding='utf-8-sig')
df3.to_csv('btc_tweet_3.csv',index=False, encoding='utf-8-sig')
df4.to_csv('btc_tweet_4.csv',index=False, encoding='utf-8-sig')
df5.to_csv('btc_tweet_5.csv',index=False, encoding='utf-8-sig')
df6.to_csv('btc_tweet_6.csv',index=False, encoding='utf-8-sig')
df7.to_csv('btc_tweet_7.csv',index=False, encoding='utf-8-sig')
df8.to_csv('btc_tweet_8.csv',index=False, encoding='utf-8-sig')
df9.to_csv('btc_tweet_9.csv',index=False, encoding='utf-8-sig')
df10.to_csv('btc_tweet_10.csv',index=False, encoding='utf-8-sig')